# 01 - Inspección Rápida del Dataset (AIDev)

Ejecuta consultas reales a la API del Dataset Viewer (Hugging Face) usando `exploration/aidev/inspect_aidev.py`.

Entrega trazabilidad en `exploration/aidev/reports/` (JSON) y puede sobrescribir archivos.

Requiere red (datasets-server).


In [ ]:
from __future__ import annotations

import json
import subprocess
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path.cwd()
PY = PROJECT_ROOT / '.venv' / 'bin' / 'python'
INSPECT = PROJECT_ROOT / 'exploration/aidev/inspect_aidev.py'
REPORTS_DIR = PROJECT_ROOT / 'exploration/aidev/reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

DATASET = 'hao-li/AIDev'
CONFIG = 'all_pull_request'
SPLIT = 'train'
PROFILE_LIMIT = 500
PREVIEW_LIMIT = 3
OVERWRITE_REPORTS = True

def run_json(cmd: list[str]) -> dict:
    print('[cmd]', ' '.join(map(str, cmd)))
    proc = subprocess.run(cmd, check=True, text=True, capture_output=True)
    return json.loads(proc.stdout)

def report_path(name: str) -> Path:
    if OVERWRITE_REPORTS:
        return REPORTS_DIR / name
    ts = datetime.utcnow().strftime('%Y-%m-%dT%H%M%SZ')
    return REPORTS_DIR / f'{ts}-{name}'

def write_report(name: str, payload: dict) -> Path:
    path = report_path(name)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')
    print('[ok] wrote', path)
    return path

print('[info] dataset=', DATASET)
print('[info] config=', CONFIG, 'split=', SPLIT)


In [ ]:
print('\n[step] 1/3 overview')
overview = run_json([str(PY), str(INSPECT), '--dataset', DATASET, 'overview'])
write_report('inspect-overview.json', overview)
print('[info] total_rows=', overview.get('total_rows'))


In [ ]:
print('\n[step] 2/3 preview')
preview = run_json([
    str(PY), str(INSPECT), '--dataset', DATASET,
    'preview', '--config', CONFIG, '--split', SPLIT, '--limit', str(PREVIEW_LIMIT)
])
write_report('inspect-preview.json', preview)
rows = preview.get('rows') or []
if rows:
    print('[info] first row keys=', list(rows[0].keys()))
print('[info] feature_count=', len(preview.get('features') or []))


In [ ]:
print('\n[step] 3/3 profile')
profile = run_json([
    str(PY), str(INSPECT), '--dataset', DATASET,
    'profile', '--config', CONFIG, '--split', SPLIT,
    '--limit', str(PROFILE_LIMIT), '--top-k', '10'
])
write_report('inspect-profile.json', profile)
summary = profile.get('summary') or {}
vc = summary.get('value_counts') or {}
print('[info] rows_seen=', summary.get('rows_seen'))
print('[info] value_counts.agent=', vc.get('agent'))
print('[info] value_counts.state=', vc.get('state'))
